In [12]:
import pandas as pd
import numpy as np

In [13]:
import pandas as pd
import math

def angle(point1, point2, point3):
    """ Calculate angle between two lines """
    # p1 = 起點，p2 = 頂點（關節處），p3 = 終點
    if(point1 == (0, 0) or point2 == (0, 0) or point3 == (0, 0)):
        return 0
    numerator = point2[1] * (point1[0] - point3[0]) + point1[1] * \
                (point3[0] - point2[0]) + point3[1] * (point2[0] - point1[0])
    denominator = (point2[0] - point1[0]) * (point1[0] - point3[0]) + \
                (point2[1] - point1[1]) * (point1[1] - point3[1])
    try:
        ang = math.atan(numerator / denominator)
        ang = ang * 180 / math.pi
        if ang < 0:
            ang = 180 + ang
        return ang
    except:
        return 90.0

data = pd.read_csv('/content/drive/MyDrive/1.csv')#demo03_1.csv
data = data.drop(columns=['z'])

In [14]:
def get_landmark_coordinates(data, side, frame_number):
    return {
        'shoulder': data[(data['landmark'] == f'{side}_SHOULDER') & (data['frame_number'] == frame_number)][['x', 'y']].values[0],
        'elbow': data[(data['landmark'] == f'{side}_ELBOW') & (data['frame_number'] == frame_number)][['x', 'y']].values[0],
        'wrist': data[(data['landmark'] == f'{side}_WRIST') & (data['frame_number'] == frame_number)][['x', 'y']].values[0],
        'hip': data[(data['landmark'] == f'{side}_HIP') & (data['frame_number'] == frame_number)][['x', 'y']].values[0],
        'knee': data[(data['landmark'] == f'{side}_KNEE') & (data['frame_number'] == frame_number)][['x', 'y']].values[0],
        'ankle': data[(data['landmark'] == f'{side}_ANKLE') & (data['frame_number'] == frame_number)][['x', 'y']].values[0]
    }

def calculate_angles(coords):
    angles = {}
    angles['shoulder-elbow-wrist'] = angle(tuple(coords['shoulder']), tuple(coords['elbow']), tuple(coords['wrist']))
    angles['shoulder-hip-knee'] = angle(tuple(coords['shoulder']), tuple(coords['hip']), tuple(coords['knee']))
    angles['hip-knee-ankle'] = angle(tuple(coords['hip']), tuple(coords['knee']), tuple(coords['ankle']))
    return angles

all_angles = []

for frame_number in data['frame_number'].unique():
    left_coords = get_landmark_coordinates(data, 'LEFT', frame_number)
    right_coords = get_landmark_coordinates(data, 'RIGHT', frame_number)

    left_angles = calculate_angles(left_coords)
    right_angles = calculate_angles(right_coords)

    for angle_type, angle_value in left_angles.items():
        all_angles.append({'frame_number': frame_number, '角度類型': f"(左){angle_type}", '角度': angle_value})

    for angle_type, angle_value in right_angles.items():
        all_angles.append({'frame_number': frame_number, '角度類型': f"(右){angle_type}", '角度': angle_value})

df = pd.DataFrame(all_angles)

df


,frame_number,角度類型,角度
0,0,(左)shoulder-elbow-wrist,9.574640
1,0,(左)shoulder-hip-knee,19.105547
2,0,(左)hip-knee-ankle,177.814304
3,0,(右)shoulder-elbow-wrist,8.188391
4,0,(右)shoulder-hip-knee,14.650039
...,...,...,...
1939,323,(左)shoulder-hip-knee,10.908544
1940,323,(左)hip-knee-ankle,179.780032
1941,323,(右)shoulder-elbow-wrist,126.650858
1942,323,(右)shoulder-hip-knee,14.922683
